# Experiment: ASR Dataset Recipe Builder for Whisper

Objective:
- Understand the schema and training relevance of the three target datasets before any curation or filtering.
- Normalize them into a shared metadata-first recipe that matches the Whisper preparation pattern in `Akan_Finetuning_Standard_specAug.ipynb`.
- Keep audio decoding and audio downloads out of scope until explicitly needed and approved.


## Scope

This notebook is for dataset preparation and exploratory curation planning, not model training.

It is designed to support these datasets:
- `fiifinketia/WaxalNLP` with subset `aka_asr`
- `fiifinketia/ghana-english-asr-2700hrs`
- `fiifinketia/twi-trigrams-speech-text-parallel`

Design constraints:
- Default to metadata-first operations.
- Do not decode audio unless explicitly enabled.
- Structure the notebook so researchers can run analysis cells independently and in any order.
- Do not implement filtering logic yet; only prepare the recipe and analysis surface.


In [ ]:
from __future__ import annotations

import math
import os
import random
import re
import statistics
import unicodedata
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

SEED = 7
random.seed(SEED)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 160)

NOTEBOOK_ROOT = Path.cwd()
OUTPUT_DIR = NOTEBOOK_ROOT / "output" / "jupyter-notebook" / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED


## Environment and loading policy

The reference Whisper notebook eventually expects:
- an `audio` column for waveform access
- a normalized text field that is tokenized into labels
- train and evaluation splits prepared before feature extraction

This notebook stops one stage earlier. It builds a canonical metadata recipe with a standard text column and enough fields to support later selection and analysis.


In [ ]:
# Install locally if needed:
# %pip install -q "datasets[audio]==2.14.5" pandas matplotlib seaborn

RECOMMENDED_DATASETS_VERSION = "2.14.5"
ALLOW_AUDIO_DECODE = False
MAX_PREVIEW_ROWS = 5
TEXT_SAMPLE_SIZE = 10_000

print({
    "recommended_datasets_version": RECOMMENDED_DATASETS_VERSION,
    "allow_audio_decode": ALLOW_AUDIO_DECODE,
    "text_sample_size": TEXT_SAMPLE_SIZE,
})


In [ ]:
import datasets
from datasets import Audio, Dataset, DatasetDict, IterableDataset, IterableDatasetDict, load_dataset

print("datasets version:", datasets.__version__)


## Dataset registry

The `schema` entry maps each source dataset into a canonical ASR recipe schema.

Canonical columns:
- `source_dataset`
- `source_subset`
- `source_split`
- `record_id`
- `speaker_id`
- `gender`
- `language`
- `text`
- `text_source_column`
- `duration_seconds`
- `audio`

`text` is the field that will later feed Whisper tokenization.


In [ ]:
DATASETS = {
    "waxal_aka_asr": {
        "path": "fiifinketia/WaxalNLP",
        "name": "aka_asr",
        "splits": ["train", "validation", "test"],
        "schema": {
            "record_id": "id",
            "speaker_id": "speaker_id",
            "gender": "gender",
            "language": "language",
            "text": "transcription",
            "duration_seconds": None,
            "audio": "audio",
        },
        "notes": "Waxal Akan ASR already resembles a Whisper-ready ASR dataset; text comes from `transcription`.",
    },
    "ghana_english_asr": {
        "path": "fiifinketia/ghana-english-asr-2700hrs",
        "name": None,
        "splits": ["train"],
        "schema": {
            "record_id": None,
            "speaker_id": None,
            "gender": None,
            "language": None,
            "text": "corrected_text",
            "duration_seconds": "duration_ss",
            "audio": "audio",
        },
        "notes": "Broadcast-news English ASR; duration metadata is already provided.",
    },
    "twi_trigrams_parallel": {
        "path": "fiifinketia/twi-trigrams-speech-text-parallel",
        "name": None,
        "splits": ["train"],
        "schema": {
            "record_id": None,
            "speaker_id": None,
            "gender": None,
            "language": None,
            "text": "text",
            "duration_seconds": None,
            "audio": "audio",
        },
        "notes": "Short Twi speech-text segments; useful as a targeted augmentation source but structurally different from sentence-level ASR corpora.",
    },
}

pd.DataFrame([
    {
        "dataset_key": key,
        "path": spec["path"],
        "subset": spec["name"],
        "splits": ", ".join(spec["splits"]),
        "text_column": spec["schema"]["text"],
        "duration_column": spec["schema"]["duration_seconds"],
        "notes": spec["notes"],
    }
    for key, spec in DATASETS.items()
])


## Metadata-first loading helpers

The goal here is to inspect schema and build metadata tables without pulling decoded waveforms.

Two controls matter:
- `streaming=True` avoids materializing the full dataset locally.
- `Audio(decode=False)` keeps the `audio` column as metadata/path references until you intentionally decode it.


In [ ]:
def load_source_dataset(dataset_key: str, split: str | None = None, streaming: bool = True):
    spec = DATASETS[dataset_key]
    kwargs = {"path": spec["path"], "streaming": streaming}
    if spec["name"] is not None:
        kwargs["name"] = spec["name"]
    if split is not None:
        kwargs["split"] = split

    ds = load_dataset(**kwargs)
    if split is not None and "audio" in getattr(ds, "features", {}):
        ds = ds.cast_column("audio", Audio(decode=ALLOW_AUDIO_DECODE))
    return ds


def peek_records(ds, n: int = MAX_PREVIEW_ROWS) -> list[dict[str, Any]]:
    rows = []
    for idx, row in enumerate(ds):
        rows.append(row)
        if idx + 1 >= n:
            break
    return rows


def get_feature_map(ds) -> dict[str, str]:
    features = getattr(ds, "features", None)
    if features is None:
        return {}
    return {key: str(value) for key, value in features.items()}


In [ ]:
schema_reports = []
for dataset_key, spec in DATASETS.items():
    for split in spec["splits"]:
        ds = load_source_dataset(dataset_key, split=split, streaming=True)
        schema_reports.append({
            "dataset_key": dataset_key,
            "path": spec["path"],
            "subset": spec["name"],
            "split": split,
            "features": get_feature_map(ds),
        })

schema_reports


## Quick schema inspection

This is the first check to confirm how each dataset maps into the recipe.


In [ ]:
for report in schema_reports:
    print(f"\n=== {report['dataset_key']} | split={report['split']} ===")
    for column, feature in report["features"].items():
        print(f"- {column}: {feature}")


In [ ]:
preview_frames = {}
for dataset_key, spec in DATASETS.items():
    split = spec["splits"][0]
    ds = load_source_dataset(dataset_key, split=split, streaming=True)
    preview = pd.DataFrame(peek_records(ds, n=MAX_PREVIEW_ROWS))
    preview_frames[dataset_key] = preview
    print(f"\nPreview for {dataset_key} ({split})")
    display(preview.head(MAX_PREVIEW_ROWS))


## Canonical recipe normalization

This section converts every source row into the same metadata shape. It does not create Whisper features yet.

That later training step would map roughly as:
- `audio` -> Whisper feature extractor -> `input_features`
- `text` -> Whisper tokenizer -> `labels`

The normalization step below is the reusable bridge between your source datasets and the training notebook.


In [ ]:
def safe_get(row: dict[str, Any], key: str | None, default: Any = None) -> Any:
    if key is None:
        return default
    return row.get(key, default)


def normalize_text(text: Any) -> str:
    if text is None:
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def infer_language(dataset_key: str, row: dict[str, Any], spec: dict[str, Any]) -> str | None:
    explicit = safe_get(row, spec["schema"]["language"])
    if explicit:
        return explicit
    defaults = {
        "ghana_english_asr": "en-GH",
        "twi_trigrams_parallel": "twi",
    }
    return defaults.get(dataset_key)


def infer_record_id(dataset_key: str, split: str, row_index: int, row: dict[str, Any], spec: dict[str, Any]) -> str:
    candidate = safe_get(row, spec["schema"]["record_id"])
    if candidate not in (None, ""):
        return str(candidate)
    return f"{dataset_key}:{split}:{row_index}"


def normalize_record(dataset_key: str, split: str, row_index: int, row: dict[str, Any]) -> dict[str, Any]:
    spec = DATASETS[dataset_key]
    schema = spec["schema"]
    text_source_column = schema["text"]
    duration_key = schema["duration_seconds"]

    return {
        "source_dataset": spec["path"],
        "source_subset": spec["name"],
        "source_split": split,
        "dataset_key": dataset_key,
        "record_id": infer_record_id(dataset_key, split, row_index, row, spec),
        "speaker_id": safe_get(row, schema["speaker_id"]),
        "gender": safe_get(row, schema["gender"]),
        "language": infer_language(dataset_key, row, spec),
        "text": normalize_text(safe_get(row, text_source_column, "")),
        "text_source_column": text_source_column,
        "duration_seconds": safe_get(row, duration_key),
        "audio": safe_get(row, schema["audio"]),
    }


def build_metadata_sample(dataset_key: str, split: str, limit: int = MAX_PREVIEW_ROWS) -> pd.DataFrame:
    ds = load_source_dataset(dataset_key, split=split, streaming=True)
    rows = []
    for idx, row in enumerate(ds):
        rows.append(normalize_record(dataset_key, split, idx, row))
        if idx + 1 >= limit:
            break
    return pd.DataFrame(rows)


In [ ]:
metadata_samples = {}
for dataset_key, spec in DATASETS.items():
    for split in spec["splits"]:
        sample_df = build_metadata_sample(dataset_key, split, limit=MAX_PREVIEW_ROWS)
        metadata_samples[(dataset_key, split)] = sample_df
        print(f"\nNormalized sample for {dataset_key} ({split})")
        display(sample_df)


## Source-to-recipe compatibility notes

Initial interpretation from the schemas:
- `WaxalNLP/aka_asr` is the closest fit to the reference Whisper pipeline because it already has `transcription`, `speaker_id`, `gender`, and predefined train/validation/test splits.
- `ghana-english-asr-2700hrs` is structurally clean for Whisper fine-tuning but currently has only a train split, so a researcher will likely need to derive evaluation splits later.
- `twi-trigrams-speech-text-parallel` is parallel audio-text data, but it is segment-level and much shorter than sentence-style ASR corpora. It is likely better treated as a supplementary dataset, not the only training source, depending on the target behavior.


In [ ]:
compatibility_notes = pd.DataFrame([
    {
        "dataset_key": "waxal_aka_asr",
        "fit_for_reference_pipeline": "High",
        "why": "Already ASR-labeled with splits and speaker metadata.",
        "expected_text_column": DATASETS["waxal_aka_asr"]["schema"]["text"],
    },
    {
        "dataset_key": "ghana_english_asr",
        "fit_for_reference_pipeline": "Medium-High",
        "why": "Whisper-compatible ASR structure, but split strategy and speaker metadata are limited.",
        "expected_text_column": DATASETS["ghana_english_asr"]["schema"]["text"],
    },
    {
        "dataset_key": "twi_trigrams_parallel",
        "fit_for_reference_pipeline": "Conditional",
        "why": "Useful as targeted augmentation or short-utterance source; not structurally equivalent to sentence-level corpora.",
        "expected_text_column": DATASETS["twi_trigrams_parallel"]["schema"]["text"],
    },
])

compatibility_notes


## Analysis toolkit

These helpers are intentionally independent. A researcher should be able to run text analysis, duration analysis, metadata audits, and later audio analysis in any order.

For now:
- text-based and metadata-based analysis is implemented
- audio-dependent analysis is declared but blocked until `ALLOW_AUDIO_DECODE=True` and explicit permission is granted


In [ ]:
def sample_normalized_rows(dataset_key: str, split: str, limit: int = TEXT_SAMPLE_SIZE) -> pd.DataFrame:
    ds = load_source_dataset(dataset_key, split=split, streaming=True)
    rows = []
    for idx, row in enumerate(ds):
        rows.append(normalize_record(dataset_key, split, idx, row))
        if idx + 1 >= limit:
            break
    return pd.DataFrame(rows)


def add_text_metrics(df: pd.DataFrame) -> pd.DataFrame:
    enriched = df.copy()
    enriched["char_count"] = enriched["text"].fillna("").str.len()
    enriched["word_count"] = enriched["text"].fillna("").str.split().str.len()
    enriched["is_empty_text"] = enriched["text"].fillna("").eq("")
    return enriched


def token_frequency(df: pd.DataFrame, top_k: int = 50) -> pd.DataFrame:
    tokens = Counter()
    for text in df["text"].fillna(""):
        tokens.update(text.lower().split())
    return pd.DataFrame(tokens.most_common(top_k), columns=["token", "count"])


def metadata_overview(df: pd.DataFrame) -> pd.Series:
    non_null_duration = df["duration_seconds"].dropna()
    return pd.Series({
        "rows": len(df),
        "empty_text_rows": int(df["text"].fillna("").eq("").sum()),
        "unique_speakers": df["speaker_id"].nunique(dropna=True),
        "known_gender_rows": int(df["gender"].notna().sum()),
        "duration_rows": int(non_null_duration.shape[0]),
        "duration_mean_seconds": float(non_null_duration.mean()) if len(non_null_duration) else math.nan,
    })


def gender_breakdown(df: pd.DataFrame) -> pd.DataFrame:
    out = df["gender"].fillna("UNKNOWN").value_counts(dropna=False).rename_axis("gender").reset_index(name="rows")
    out["share"] = out["rows"] / max(len(df), 1)
    return out


In [ ]:
analysis_frames = {}
for dataset_key, spec in DATASETS.items():
    split = spec["splits"][0]
    df = add_text_metrics(sample_normalized_rows(dataset_key, split, limit=TEXT_SAMPLE_SIZE))
    analysis_frames[dataset_key] = df

    print(f"\n=== Analysis snapshot: {dataset_key} | split={split} ===")
    display(metadata_overview(df).to_frame("value"))
    display(gender_breakdown(df))
    display(token_frequency(df, top_k=20).head(20))


## Duration and clip-length analysis

This works only where duration metadata already exists. For datasets without a duration column, leave it blank for now and fill it later only if audio decoding is approved.


In [ ]:
def duration_summary(df: pd.DataFrame) -> pd.Series:
    values = pd.to_numeric(df["duration_seconds"], errors="coerce").dropna()
    if values.empty:
        return pd.Series({"rows_with_duration": 0})
    return pd.Series({
        "rows_with_duration": int(values.shape[0]),
        "min_seconds": float(values.min()),
        "median_seconds": float(values.median()),
        "mean_seconds": float(values.mean()),
        "max_seconds": float(values.max()),
    })

for dataset_key, df in analysis_frames.items():
    print(f"\nDuration summary for {dataset_key}")
    display(duration_summary(df).to_frame("value"))


## Audio-dependent analysis placeholder

Examples that belong here later:
- noise profile estimation
- silence ratio
- clipping / loudness issues
- speech rate estimates from waveform + transcript

These analyses should only run after explicit approval because they may require fetching and decoding large audio assets.


In [ ]:
def require_audio_permission() -> None:
    if not ALLOW_AUDIO_DECODE:
        raise RuntimeError(
            "Audio decoding is disabled. Set ALLOW_AUDIO_DECODE=True only after explicit approval to download/decode audio."
        )


def audio_quality_analysis_placeholder(*args, **kwargs):
    require_audio_permission()
    raise NotImplementedError("Audio quality analysis is intentionally deferred.")


print("Audio analysis status: deferred until permission is granted.")


## Recipe object for later curation

This is the notebook artifact a researcher can edit before filters are introduced.

The idea is:
- define which datasets participate
- define which split strategy to keep or derive
- define which analyses to run
- later add filters and thresholds based on those analyses


In [ ]:
recipe = {
    "task": "asr_whisper_preparation",
    "audio_decode_enabled": ALLOW_AUDIO_DECODE,
    "canonical_text_column": "text",
    "canonical_audio_column": "audio",
    "datasets": [
        {
            "dataset_key": key,
            "path": spec["path"],
            "subset": spec["name"],
            "splits": spec["splits"],
            "schema": spec["schema"],
        }
        for key, spec in DATASETS.items()
    ],
    "analysis_modules": [
        "schema_inspection",
        "text_frequency",
        "text_length",
        "gender_breakdown",
        "duration_summary",
        "noise_profile_placeholder",
    ],
    "filtering": {
        "status": "not_started",
        "notes": "Implement after schema agreement and analysis review.",
    },
}

recipe


In [ ]:
recipe_path = OUTPUT_DIR / "asr_dataset_recipe.json"
with recipe_path.open("w", encoding="utf-8") as f:
    json_ready = recipe.copy()
    import json
    json.dump(json_ready, f, indent=2)

print("Wrote", recipe_path)


## Recommended next implementation step

Do next, but not in this notebook yet unless requested:
- Add pluggable filters that operate on the canonical metadata schema.
- Keep filters composable so researchers can choose any order.
- Separate metadata-only filters from audio-dependent filters.

Examples of future filter modules:
- transcript length filter
- duplicate transcript filter
- duration band filter
- speaker balance filter
- gender balance filter
- noise threshold filter
- language/script normalization filter


## Validation checklist

Before using this recipe in a training notebook:
- Confirm the selected source dataset still exposes the expected columns.
- Confirm the final chosen text normalization policy.
- If using Ghana English or Twi trigram data for training, define an evaluation split strategy.
- Only after approval, enable audio decoding and compute waveform-dependent features or quality metrics.
